<a href="https://colab.research.google.com/github/Hejran-M/Adversarial-defense-for-battery-state-of-health-prediction-models/blob/main/fgsm_attack_and_adverserial_training20240218_epsilon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv

import sys
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn import metrics

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D  # Updated import statement
from tensorflow.keras.utils import plot_model
from keras import backend as K
from keras.models import load_model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/T25 (1).csv')

In [ ]:
set(df['Dataset'].tolist())

In [ ]:
df.columns

In [ ]:
X = [eval(i) for i in df['resized_scaled'].tolist()]
X = np.array(X)
y = np.array(df['SOH'].tolist()) ##%changed

In [ ]:
# labeling for stratified random sampling
label = df['SOH'].tolist()
label_l = []
for cnt in range(len(label)):
    i = label[cnt]
    if i>=1.0 :
        lab = '1~'
        num = 0
    elif 0.98<=i<1.0 :
        lab = '0.98~1'
        num = 1
    elif 0.96<=i<0.98 :
        lab = '0.96~0.98'
        num = 2
    elif 0.94<=i<0.96 :
        lab = '0.94~0.96'
        num = 3
    elif 0.92<=i<0.94 :
        lab = '0.92~0.94'
        num = 4
    elif 0.90<=i<0.92 :
        lab = '0.90~0.92'
        num = 5

    elif 0.88<=i<0.9 :
        lab = '0.88~0.9'
        num = 6
    elif 0.86<=i<0.88 :
        lab = '0.86~0.88'
        num = 7
    elif 0.84<=i<0.86 :
        lab = '0.84~0.86'
        num = 8
    elif 0.82<=i<0.84 :
        lab = '0.82~0.84'
        num = 9
    elif 0.80<=i<0.82 :
        lab = '0.80~0.82'
        num = 10


    elif 0.78<=i<0.8 :
        lab = '0.78~0.8'
        num = 11
    elif 0.76<=i<0.78 :
        lab = '0.76~0.78'
        num = 12
    elif 0.74<=i<0.76 :
        lab = '0.74~0.76'
        num = 13
    elif 0.72<=i<0.74 :
        lab = '0.72~0.74'
        num = 14
    elif 0.70<=i<0.72 :
        lab = '0.70~0.72'
        num = 15

    else:
        lab = '~0.7'
        num = 16
    label_l.append(num)

In [ ]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx in sss.split(X, np.array(label_l)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]
    y_valid = np.array(label_l)[test_idx]

In [ ]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx1 in sss.split(X_test, y_valid):
    X_valid, X_test_f = X_test[train_idx], X_test[test_idx1]
    y_valid, y_test_f = np.array(y_test)[train_idx], np.array(y_test)[test_idx1]
X_valid.shape

In [ ]:
data_list = []
for i in df['Dataset'][test_idx[test_idx1]].tolist():
    if 'LFP' in i:
        data_list.append('LFP')
    if 'NCA' in i:
        data_list.append('NCA')
    if 'NMC' in i:
        data_list.append('NMC')

In [ ]:
x_train = X_train
x_valid = X_valid
x_test = X_test_f

In [ ]:
rows = len(X[0])
cols = 3
input_shape = (rows, cols, 1)
x_train = x_train.reshape(x_train.shape[0], rows, cols, 1)
x_test = x_test.reshape(x_test.shape[0], rows, cols, 1)
x_valid = x_valid.reshape(x_valid.shape[0], rows, cols, 1)

x_train = x_train.astype('float32') /255.0
x_test = x_test.astype('float32')/255.0
x_valid = x_valid.astype('float32')/255.0

batch_size = 64
x_train.shape

In [ ]:
###
model = Sequential()
model.add(Conv2D(32, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(64, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(MaxPooling2D(pool_size = (2,2), strides = (1,1)))
model.add(Flatten())
model.add(Dense(64,activation = 'relu'))
model.add(Dense(1))
model.summary()
###

In [ ]:
start = time.time()
epochs =5
print('start time : ' , start)
model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mse'])
hist2 = model.fit(x_train, y_train, epochs = epochs, verbose = 1, validation_data = (x_valid, y_valid))

In [ ]:
y_vloss = hist2.history['val_loss']
y_loss = hist2.history['loss']
x_len = np.arange(len(y_loss))

plt.plot(x_len, y_vloss, marker='.', c='red', label="Validation-set Loss")
plt.plot(x_len, y_loss, marker='.', c='blue', label="Train-set Loss")
plt.legend()
plt.title('epoch40_loss')

In [ ]:
predict = model.predict(x_test, verbose = 0)

In [ ]:
p = []
t = []
for i in range(len(predict)):
    p.append(predict[i])
    t.append(y_test_f[i])
    #print('predicted SOH: ' , predict[i],',' , 'Answer SOH: '  , y_test[i])

plt.scatter(y_test_f, predict)
plt.plot(y_test_f, y_test_f,c='red')
plt.show()

In [ ]:
r2 = metrics.r2_score(y_test_f, predict)
rmse = metrics.mean_squared_error(y_test_f, predict)**0.5
mse = metrics.mean_squared_error(y_test_f, predict)
mae = metrics.mean_absolute_error(y_test_f, predict)
mape = metrics.mean_absolute_percentage_error(y_test_f,predict)

In [ ]:
dic = {}
dic['r2'] = [r2]
dic['rmse'] = [rmse]
dic['mse'] = [mse]
dic['mae'] = [mae]
dic['mape'] = [mape]
d = pd.DataFrame(dic)
d

In [ ]:
def fgsm_perturb(x_test, epsilon):
    x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_test_tf)
        output = model(x_test_tf)
        loss = tf.keras.losses.mean_squared_error(y_test_f, output)
    # Calculate gradient of the loss with respect to the input
    gradient = tape.gradient(loss, x_test_tf)

    # Create perturbation using the sign of the gradient
    perturbation = epsilon * tf.sign(gradient)

    # Apply perturbation to the input data
    perturbed_data = x_test_tf + perturbation

    return perturbed_data.numpy()


In [ ]:
# List of epsilon values to test
epsilon_values = [i / 100 for i in range(1, 3)]

# Dictionary to store evaluation results for different epsilon values
epsilon_results = {'Epsilon': [], 'Attack': [], 'True SOH': [], 'Predicted SOH': [], 'r2': [], 'rmse': [], 'mse': [], 'mae': [], 'mape': []}

# Loop through each epsilon value
for epsilon in epsilon_values:
    # Perturb the test data
    x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)

    original_predictions = model.predict(x_test, verbose=0)
    perturbed_predictions_fgsm = model.predict(x_test_ad_fgsm, verbose=0)


    predict_fgsm = perturbed_predictions_fgsm


    r2 = metrics.r2_score(y_test_f, original_predictions)
    rmse = metrics.mean_squared_error(y_test_f, original_predictions) ** 0.5
    mse = metrics.mean_squared_error(y_test_f, original_predictions)
    mae = metrics.mean_absolute_error(y_test_f, original_predictions)
    mape = metrics.mean_absolute_percentage_error(y_test_f, original_predictions)

    r2_fgsm = metrics.r2_score(y_test_f,  predict_fgsm)
    rmse_fgsm = metrics.mean_squared_error(y_test_f,  predict_fgsm) ** 0.5
    mse_fgsm = metrics.mean_squared_error(y_test_f, predict_fgsm)
    mae_fgsm = metrics.mean_absolute_error(y_test_f,  predict_fgsm)
    mape_fgsm = metrics.mean_absolute_percentage_error(y_test_f,  predict_fgsm)



    # Store results in the dictionary
    epsilon_results['Epsilon'].extend([epsilon]*3)
    epsilon_results['Attack'].extend(['FGSM', 'IFGSM', 'MIFGSM'])
    epsilon_results['True SOH'].extend([y_test_f]*3)
    epsilon_results['Predicted SOH'].extend([predict_fgsm])
    epsilon_results['r2'].extend([r2_fgsm])
    epsilon_results['rmse'].extend([rmse_fgsm])
    epsilon_results['mse'].extend([mse_fgsm])
    epsilon_results['mae'].extend([mae_fgsm])
    epsilon_results['mape'].extend([mape_fgsm])
    plt.scatter(y_test_f, predict_fgsm, label='FGSM')
    plt.plot(y_test_f, y_test_f, c='red', label='True SOH')
    plt.title(f'Predictions with Epsilon = {epsilon}')
    plt.xlabel('True SOH')
    plt.ylabel('Predicted SOH')
    plt.legend()
    plt.show()

# Create a DataFrame to display the results for different epsilon values



In [ ]:
# Calculate the number of samples for the attack
attack_samples = int(0.2 * len(x_train))  # 20% of the training data

# Randomly select indices for the attack
attack_indices = np.random.choice(len(x_train), attack_samples, replace=False)

# Generate adversarial examples for the selected subset using FGSM
epsilon = 0.01  # Adjust the value of epsilon as needed
x_train_attack = fgsm_perturb(x_train[attack_indices], epsilon)
# Append the adversarial examples to the original training data
x_train_new = np.concatenate((x_train, x_train_attack), axis=0)
y_train_new = np.concatenate((y_train, y_train[attack_indices]), axis=0)

# Rest of your code...
x_train = x_train_new
y_train = y_train_new

In [ ]:
###
model = Sequential()
model.add(Conv2D(32, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(64, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(MaxPooling2D(pool_size = (2,2), strides = (1,1)))
model.add(Flatten())
model.add(Dense(64,activation = 'relu'))
model.add(Dense(1))
model.summary()
###

In [ ]:
start = time.time()
epochs = 5
print('start time : ' , start)
model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mse'])
hist2 = model.fit(x_train, y_train, epochs = epochs, verbose = 1, validation_data = (x_valid, y_valid))

In [ ]:
y_vloss = hist2.history['val_loss']
y_loss = hist2.history['loss']
x_len = np.arange(len(y_loss))

plt.plot(x_len, y_vloss, marker='.', c='red', label="Validation-set Loss")
plt.plot(x_len, y_loss, marker='.', c='blue', label="Train-set Loss")
plt.legend()
plt.title('epoch40_loss')

In [ ]:
# List of epsilon values to test
epsilon_values = [i / 100 for i in range(1, 3)]

# Dictionary to store evaluation results for different epsilon values
epsilon_results = {'Epsilon': [], 'Attack': [], 'True SOH': [], 'Predicted SOH': [], 'r2': [], 'rmse': [], 'mse': [], 'mae': [], 'mape': []}

# Loop through each epsilon value
for epsilon in epsilon_values:
    # Perturb the test data
    x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)

    original_predictions = model.predict(x_test, verbose=0)
    perturbed_predictions_fgsm = model.predict(x_test_ad_fgsm, verbose=0)


    predict_fgsm = perturbed_predictions_fgsm


    r2 = metrics.r2_score(y_test_f, original_predictions)
    rmse = metrics.mean_squared_error(y_test_f, original_predictions) ** 0.5
    mse = metrics.mean_squared_error(y_test_f, original_predictions)
    mae = metrics.mean_absolute_error(y_test_f, original_predictions)
    mape = metrics.mean_absolute_percentage_error(y_test_f, original_predictions)

    r2_fgsm = metrics.r2_score(y_test_f,  predict_fgsm)
    rmse_fgsm = metrics.mean_squared_error(y_test_f,  predict_fgsm) ** 0.5
    mse_fgsm = metrics.mean_squared_error(y_test_f, predict_fgsm)
    mae_fgsm = metrics.mean_absolute_error(y_test_f,  predict_fgsm)
    mape_fgsm = metrics.mean_absolute_percentage_error(y_test_f,  predict_fgsm)



    # Store results in the dictionary
    epsilon_results['Epsilon'].extend([epsilon]*3)
    epsilon_results['Attack'].extend(['FGSM', 'IFGSM', 'MIFGSM'])
    epsilon_results['True SOH'].extend([y_test_f]*3)
    epsilon_results['Predicted SOH'].extend([predict_fgsm])
    epsilon_results['r2'].extend([r2_fgsm])
    epsilon_results['rmse'].extend([rmse_fgsm])
    epsilon_results['mse'].extend([mse_fgsm])
    epsilon_results['mae'].extend([mae_fgsm])
    epsilon_results['mape'].extend([mape_fgsm])
    plt.scatter(y_test_f, predict_fgsm, label='FGSM')
    plt.plot(y_test_f, y_test_f, c='red', label='True SOH')
    plt.title(f'Predictions with Epsilon = {epsilon}')
    plt.xlabel('True SOH')
    plt.ylabel('Predicted SOH')
    plt.legend()
    plt.show()

In [ ]:
# List of epsilon values to test
epsilon_values = [i / 100 for i in range(1, 3)]

# Dictionary to store evaluation results for different epsilon values
epsilon_results = {'Epsilon': [], 'Attack': [], 'True SOH': [], 'Predicted SOH': [], 'r2': [], 'rmse': [], 'mse': [], 'mae': [], 'mape': []}

# Loop through each epsilon value
for epsilon in epsilon_values:
    # Perturb the test data
    x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)
    x_test_ad_ifgsm = ifgsm_attack(x_test, epsilon)  # You need to calculate data_grad somehow
    x_test_ad_mifgsm = mifgsm_attack(x_test, epsilon)

    original_predictions = model.predict(x_test, verbose=0)
    perturbed_predictions_fgsm = model.predict(x_test_ad_fgsm, verbose=0)
    perturbed_predictions_ifgsm = model.predict(x_test_ad_ifgsm, verbose=0)
    perturbed_predictions_mifgsm = model.predict(x_test_ad_mifgsm, verbose=0)

    predict_fgsm = perturbed_predictions_fgsm
    predict_ifgsm = perturbed_predictions_ifgsm
    predict_mifgsm = perturbed_predictions_mifgsm

    r2 = metrics.r2_score(y_test_f, original_predictions)
    rmse = metrics.mean_squared_error(y_test_f, original_predictions) ** 0.5
    mse = metrics.mean_squared_error(y_test_f, original_predictions)
    mae = metrics.mean_absolute_error(y_test_f, original_predictions)
    mape = metrics.mean_absolute_percentage_error(y_test_f, original_predictions)

    r2_fgsm = metrics.r2_score(y_test_f,  predict_fgsm)
    rmse_fgsm = metrics.mean_squared_error(y_test_f,  predict_fgsm) ** 0.5
    mse_fgsm = metrics.mean_squared_error(y_test_f, predict_fgsm)
    mae_fgsm = metrics.mean_absolute_error(y_test_f,  predict_fgsm)
    mape_fgsm = metrics.mean_absolute_percentage_error(y_test_f,  predict_fgsm)

    r2_ifgsm = metrics.r2_score(y_test_f, predict_ifgsm)
    rmse_ifgsm = metrics.mean_squared_error(y_test_f, predict_ifgsm) ** 0.5
    mse_ifgsm = metrics.mean_squared_error(y_test_f, predict_ifgsm)
    mae_ifgsm = metrics.mean_absolute_error(y_test_f, predict_ifgsm)
    mape_ifgsm = metrics.mean_absolute_percentage_error(y_test_f, predict_ifgsm)

    r2_mifgsm = metrics.r2_score(y_test_f, predict_mifgsm)
    rmse_mifgsm = metrics.mean_squared_error(y_test_f, predict_mifgsm) ** 0.5
    mse_mifgsm = metrics.mean_squared_error(y_test_f, predict_mifgsm)
    mae_mifgsm = metrics.mean_absolute_error(y_test_f, predict_mifgsm)
    mape_mifgsm = metrics.mean_absolute_percentage_error(y_test_f, predict_mifgsm)

    # Store results in the dictionary
    epsilon_results['Epsilon'].extend([epsilon]*3)
    epsilon_results['Attack'].extend(['FGSM', 'IFGSM', 'MIFGSM'])
    epsilon_results['True SOH'].extend([y_test_f]*3)
    epsilon_results['Predicted SOH'].extend([predict_fgsm, predict_ifgsm, predict_mifgsm])
    epsilon_results['r2'].extend([r2_fgsm, r2_ifgsm, r2_mifgsm])
    epsilon_results['rmse'].extend([rmse_fgsm, rmse_ifgsm, rmse_mifgsm])
    epsilon_results['mse'].extend([mse_fgsm, mse_ifgsm, mse_mifgsm])
    epsilon_results['mae'].extend([mae_fgsm, mae_ifgsm, mae_mifgsm])
    epsilon_results['mape'].extend([mape_fgsm, mape_ifgsm, mape_mifgsm])

# Create a DataFrame to display the results for different epsilon values
epsilon_results_df = pd.DataFrame(epsilon_results)
print(epsilon_results_df)
# Save results to Excel
epsilon_results_df.to_excel("epsilon_results.xlsx", index=False)

In [ ]:
# List of epsilon values to test
epsilon_values = [0.03,0.2,1]
#epsilon_values = [0.01, 0.02, 0.03,0.04,0.05,0.06,0.07,0.08,0.09,0.10,0.11,0.12,0.13,0.14,0.15,0.16,0.17,0.18,0.19,0.20,0.21,0.22,0.23,0.24,0.25,0.26,0.27,0.28,0.29,0.3,0.31,0.32,0.33,0.34,0.35,0.36,0.37,0.38,0.39,0.4,0.41,0.42,0.43,0.44,0.45,0.46,0.47,0.48,0.49,0.5]  # Add more values as needed

# Dictionary to store evaluation results for different epsilon values
epsilon_results = {}

# Loop through each epsilon value
for epsilon in epsilon_values:
    # Perturb the test data
    x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)
    x_test_ad_ifgsm = ifgsm_attack(x_test, epsilon)  # You need to calculate data_grad somehow
    x_test_ad_mifgsm = mifgsm_attack(x_test, epsilon)

    original_predictions = model.predict(x_test, verbose=0)
    perturbed_predictions_fgsm = model.predict(x_test_ad_fgsm, verbose=0)
    perturbed_predictions_ifgsm = model.predict(x_test_ad_ifgsm, verbose=0)
    perturbed_predictions_mifgsm = model.predict(x_test_ad_mifgsm, verbose=0)

    predict_fgsm = perturbed_predictions_fgsm
    predict_ifgsm = perturbed_predictions_ifgsm  # Define predict_ifgsm before using it in metrics calculations
    predict_mifgsm = perturbed_predictions_mifgsm  # Define predict_mifgsm before using it in metrics calculations

    r2 = metrics.r2_score(y_test_f, original_predictions)
    rmse = metrics.mean_squared_error(y_test_f, original_predictions) ** 0.5
    mse = metrics.mean_squared_error(y_test_f, original_predictions)
    mae = metrics.mean_absolute_error(y_test_f, original_predictions)
    mape = metrics.mean_absolute_percentage_error(y_test_f, original_predictions)

    r2_fgsm = metrics.r2_score(y_test_f,  predict_fgsm)
    rmse_fgsm = metrics.mean_squared_error(y_test_f,  predict_fgsm) ** 0.5
    mse_fgsm = metrics.mean_squared_error(y_test_f, predict_fgsm)
    mae_fgsm = metrics.mean_absolute_error(y_test_f,  predict_fgsm)
    mape_fgsm = metrics.mean_absolute_percentage_error(y_test_f,  predict_fgsm)

    r2_ifgsm = metrics.r2_score(y_test_f, predict_ifgsm)
    rmse_ifgsm = metrics.mean_squared_error(y_test_f, predict_ifgsm) ** 0.5
    mse_ifgsm = metrics.mean_squared_error(y_test_f, predict_ifgsm)
    mae_ifgsm = metrics.mean_absolute_error(y_test_f, predict_ifgsm)
    mape_ifgsm = metrics.mean_absolute_percentage_error(y_test_f, predict_ifgsm)

    r2_mifgsm = metrics.r2_score(y_test_f, predict_mifgsm)
    rmse_mifgsm = metrics.mean_squared_error(y_test_f, predict_mifgsm) ** 0.5
    mse_mifgsm = metrics.mean_squared_error(y_test_f, predict_mifgsm)
    mae_mifgsm = metrics.mean_absolute_error(y_test_f, predict_mifgsm)
    mape_mifgsm = metrics.mean_absolute_percentage_error(y_test_f, predict_mifgsm)
    # Store results in the dictionary
    epsilon_results[epsilon] = {'FGSM': [r2_fgsm, rmse_fgsm, mse_fgsm, mae_fgsm, mape_fgsm],
                                'IFGSM': [r2_ifgsm, rmse_ifgsm, mse_ifgsm, mae_ifgsm, mape_ifgsm],
                                'MIFGSM': [r2_mifgsm, rmse_mifgsm, mse_mifgsm, mae_mifgsm, mape_mifgsm]}

    # Plot the predictions
    plt.scatter(y_test_f, predict_fgsm, label='FGSM')
    plt.scatter(y_test_f, predict_ifgsm, label='IFGSM')
    plt.scatter(y_test_f, predict_mifgsm, label='MIFGSM')
    plt.plot(y_test_f, y_test_f, c='red', label='True SOH')
    plt.title(f'Predictions with Epsilon = {epsilon}')
    plt.xlabel('True SOH')
    plt.ylabel('Predicted SOH')
    plt.legend()
    plt.show()

# Create a DataFrame to display the results for different epsilon values
epsilon_results_df = pd.DataFrame.from_dict(epsilon_results, orient='index')
print(epsilon_results_df)